In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

## Connecting the environment

In [ ]:
import sqlite3
import pandas as pd

path = "/kaggle/input/datasets/rtatman/188-million-us-wildfires/FPA_FOD_20170508.sqlite"

conn = sqlite3.connect(path)

## Loading data

In [ ]:
fires = pd.read_sql_query(
    "SELECT * FROM Fires LIMIT 5000",
    conn
)

list(fires.columns)
fires.info()

# 1. How many fires occurred each year?

In [ ]:
pd.set_option('display.max_rows', 20)

query1 = pd.read_sql_query(
    "SELECT FIRE_YEAR FROM Fires",
    conn
)

query1.value_counts()

# 2. Which states had the most reported fires? (Top 10)

In [ ]:
pd.set_option('display.max_rows', 10)

query2 = pd.read_sql_query(
    "SELECT STATE FROM Fires",
    conn
)

query2.groupby('STATE').STATE.count().sort_values(ascending=False).head(10)

# 3. What were the 20 largest fires in the dataset?

In [ ]:
pd.set_option('display.max_rows', 20)

query3 = pd.read_sql_query(
    """SELECT
    FIRE_NAME,
    STATE,
    FIRE_SIZE
    FROM Fires""",
    conn
)

query3.sort_values(by = 'FIRE_SIZE', ascending=False).head(20)

# 4. Which causes appear most frequently?

In [ ]:
pd.set_option('display.max_rows', 20)

query4 = pd.read_sql_query(
    "SELECT STAT_CAUSE_DESCR FROM Fires",
    conn
)

count_cause = query4.value_counts(normalize=True).mul(100).round(2)

count_cause.sort_values(ascending=False).head(5)

# 5. Average area burned by state

In [ ]:
pd.set_option('display.max_rows', 20)

query5 = pd.read_sql_query(
    """SELECT 
    STATE,
    FIRE_SIZE
    FROM Fires""",
    conn
)

avg = query5.groupby('STATE').FIRE_SIZE.mean()

avg.sort_values(ascending=False)

# 6. Fires occur by year and month

In [ ]:
pd.set_option('display.max_rows', None)

query6 = pd.read_sql_query( 
    """ SELECT 
    DISCOVERY_DATE 
    FROM Fires""", 
    conn ) 

query6['DISCOVERY_DATE'] = pd.to_datetime(query6['DISCOVERY_DATE'], origin='julian', unit='D') 

query6['DISCOVERY_MONTH'] = query6['DISCOVERY_DATE'].dt.month #for sorting

query6['DISCOVERY_MONTH_NAME'] = query6['DISCOVERY_DATE'].dt.month_name() #for viewing only

query6['DISCOVERY_YEAR'] = query6['DISCOVERY_DATE'].dt.year 

query6.groupby(['DISCOVERY_YEAR', 'DISCOVERY_MONTH','DISCOVERY_MONTH_NAME']).DISCOVERY_MONTH_NAME.size().droplevel(1, axis=0) 

# 7. How many fires covered more than 5,000 acres?

In [ ]:
pd.set_option('display.max_rows', 20)

query7 = pd.read_sql_query(
    """
    SELECT
    FIRE_SIZE
    FROM Fires""",
    conn
)

(query7.FIRE_SIZE > 5000).sum()

# 8. Which columns have the most null values?

In [ ]:
pd.set_option('display.max_rows', 20)

query8 = pd.read_sql_query(
    """
    SELECT 
    FIRE_YEAR,
    DISCOVERY_DATE,
    STAT_CAUSE_DESCR,
    FIRE_SIZE,
    FIRE_SIZE_CLASS,
    CONT_DATE,
    LATITUDE,
    LONGITUDE,
    STATE,
    COUNTY,
    OWNER_DESCR,
    FIRE_NAME,
    DISCOVERY_TIME,
    NWCG_REPORTING_AGENCY,
    SOURCE_SYSTEM_TYPE
    FROM Fires
    """,
    conn
)

pd.isnull(query8).sum().sort_values(ascending=False)

# 9. Which states take the longest to bring fires under control?

In [ ]:
pd.set_option('display.max_rows', None)

query9 = pd.read_sql_query(
    """
    SELECT 
    DISCOVERY_DATE,
    CONT_DATE,
    STATE
    FROM Fires""",
    conn
)

query9['DISCOVERY_DATE'] = pd.to_datetime(query9['DISCOVERY_DATE'], origin='julian', unit='D')
query9['CONT_DATE'] = pd.to_datetime(query9['CONT_DATE'], origin='julian', unit='D')
query9['TIME_TO_CONTROL'] = query9['CONT_DATE'] - query9['DISCOVERY_DATE']

query9_clean = query9.dropna(axis=0, how='any', subset=None, inplace=False)

query9_clean.groupby('STATE').TIME_TO_CONTROL.mean().sort_values(ascending=False)

# 10. What is the most common cause in each state?

In [ ]:
pd.set_option('display.max_rows', None)

query10 = pd.read_sql_query(
    """
    SELECT
    STATE,
    STAT_CAUSE_DESCR
    FROM Fires""",
    conn
)

query10_sorted = query10.groupby('STATE').STAT_CAUSE_DESCR.value_counts().sort_values(ascending=False).reset_index()

query10_sorted.drop_duplicates(subset='STATE', keep='first', inplace=False, ignore_index=True)